# Loan Approval Classification
This notebook implements and compares two machine learning models for predicting loan approval:

- Decision Tree
- Logistic Regression

In [1]:
# importing libraries
import pandas as pd
import numpy as np


# import train_test_split for dividing the data into training and testing sets
# import GridSearchCV for hyperparameter tuning using cross validation
from sklearn.model_selection import train_test_split, GridSearchCV

# import DecisionTreeClassifier for building the decision tree model
from sklearn.tree import DecisionTreeClassifier

# import StandardScaler for scaling features for Logistic Regression
#mean = 0 sde = 1
from sklearn.preprocessing import StandardScaler

# import LogisticRegression for building the Logistic Regression model
from sklearn.linear_model import LogisticRegression

# import evaluation metrics for measuring model performance
# accuracy_score measures overall prediction accuracy
# classification_report provides precision, recall, and F1 score
# confusion_matrix shows prediction outcomes in matrix form
# precision_score measures how many predicted positives were correct
# recall_score measures how many actual positives were correctly detected
# f1_score balances precision and recall
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score


In [2]:
# load dataset
df = pd.read_csv("loan_data.csv")

# display first 5 rows 
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
# print shape - # of rows and columns
#rows = loan applications col = features
print("Dataset shape:", df.shape)

# print all column names
print("\nColumn names:")
print(df.columns)


Dataset shape: (45000, 14)

Column names:
Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='str')


In [4]:
# print info about data types and non-null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  str    
 2   person_education                45000 non-null  str    
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  str    
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  str    
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  45000 non-n

In [5]:
# print the count of each class in the target variable loan_status
#shows how many approved vs rejected
print(df["loan_status"].value_counts())

# print the proportion of each class in the target variable to detect any imbalances
print(df["loan_status"].value_counts(normalize=True))

loan_status
0    35000
1    10000
Name: count, dtype: int64
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64


In [6]:
# check the number of missing values in each column
print(df.isnull().sum())

# check the total number of missing values in the dataset
print("Total missing values:", df.isnull().sum().sum())

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64
Total missing values: 0


In [7]:
# display summary statistics for the age column
#helps detect unrealistic values
print(df["person_age"].describe())

count    45000.000000
mean        27.764178
std          6.045108
min         20.000000
25%         24.000000
50%         26.000000
75%         30.000000
max        144.000000
Name: person_age, dtype: float64


In [8]:
#store original dataset shape before removing unrealistic ages
original_shape = df.shape

# keep only rows where person_age is less than or equal to 100
df = df[df["person_age"] <= 100]

# print the dataset shape before and after removing unrealistic ages
print("Original dataset shape:", original_shape)
print("Dataset shape after removing unrealistic ages:", df.shape)

Original dataset shape: (45000, 14)
Dataset shape after removing unrealistic ages: (44993, 14)


In [9]:
# create the feature matrix X by removing the target column loan_status
# X contains all predictor vairables used to train the models
X = df.drop("loan_status", axis=1)

#create the target vector y using the loan_status column. y represents the outcome we want to predict
#loan_status typically indicates whether a loan is approved (1) or rejected (0)
y = df["loan_status"]

In [10]:
# convert categorical variables into dummy variables
#ml algorithms require numeric input
# drop_first=True avoids the dummy variable trap by removing redundant columns
X = pd.get_dummies(X, drop_first=True)

# display the first five rows of the encoded feature matrix
#confirms that categorical features were successfully converted
X.head()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,person_gender_male,person_education_Bachelor,...,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_Yes
0,22.0,71948.0,0,35000.0,16.02,0.49,3.0,561,False,False,...,True,False,False,True,False,False,False,True,False,False
1,21.0,12282.0,0,1000.0,11.14,0.08,2.0,504,False,False,...,False,False,True,False,True,False,False,False,False,True
2,25.0,12438.0,3,5500.0,12.87,0.44,3.0,635,False,False,...,False,False,False,False,False,False,True,False,False,False
3,23.0,79753.0,0,35000.0,15.23,0.44,2.0,675,False,True,...,False,False,False,True,False,False,True,False,False,False
4,24.0,66135.0,1,35000.0,14.27,0.53,4.0,586,True,False,...,True,False,False,True,False,False,True,False,False,False


In [11]:
# split the dataset into training and testing sets using stratification.
# 80% of data = training; 20% testing
# random_state=10 ensures reproducibility of results
# stratify=y ensures both sets maintain the same class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=10,
    stratify=y
)

# print the shapes of the training and testing sets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (35994, 22)
X_test shape: (8999, 22)
y_train shape: (35994,)
y_test shape: (8999,)


In [12]:
# create a StandardScaler object for feature scaling
scaler = StandardScaler()

# fit scaler on training data and transform the training data
# fit calculates the mean and standard deviation
# transform applies standardization to the training features
X_train_scaled = scaler.fit_transform(X_train)

# transform the test features using the same fitted scaler
# prevents information leakage from the test set
X_test_scaled = scaler.transform(X_test)

## Decision Tree Model

In [13]:
# create the baseline decision tree model
#random_state ensures reproducibility of results
dt_model = DecisionTreeClassifier(random_state=10)

# train the baseline decision tree model using the training data
# tree learns decision rules that separate approved vs rejected loans
dt_model.fit(X_train, y_train)

# predict the loan status values for the test set
y_pred_dt = dt_model.predict(X_test)

In [14]:
# calculate and print the baseline decision tree accuracy
print("Baseline Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

# print the classification report containing precision, recall, and F1-score
print("\nBaseline Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

# print the confusion matrix which shows prediction outcomes
print("\nBaseline Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Baseline Decision Tree Accuracy: 0.9051005667296367

Baseline Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      6999
           1       0.78      0.80      0.79      2000

    accuracy                           0.91      8999
   macro avg       0.86      0.87      0.86      8999
weighted avg       0.91      0.91      0.91      8999


Baseline Decision Tree Confusion Matrix:
[[6554  445]
 [ 409 1591]]


In [15]:
# define the hyperparameter grid for tuning the decision tree
dt_param_grid = {
    # criterion determines how the quality of a split is measured
    # gini = gini impurity
    # entropy = information gain
    "criterion": ["gini", "entropy"],
    # max_depth controls the maximum number of levels (depth) in the tree
    # smaller values create simpler trees that reduce overfitting
    # None allows unlimited growth
    "max_depth": [3, 5, 10, None],
    # min_samples_split specifies the mininum number of samples required to split a node
    # larger values make the model more conservative and help prevent overfitting
    "min_samples_split": [2, 5, 10],
    # min_samples_leaf specifies the minimum samples required at a leaf node
    # increasing this value can help reduce overfitting
    "min_samples_leaf": [1, 2, 4],
    # max_features controls how many features the tree considers when looking for the best split
    # None - use all available features
    # sqrt - use the square root of the total number of features
    # log2 - use the base-2 logarithm of the total number of features
    # limiting features can sometimes reduce overfitting and improve generalization
    "max_features": [None, "sqrt", "log2"]
}

# create the GridSearchCV object for hyperparameter tuning
dt_grid = GridSearchCV(
    # estimator is the base decision tree model
    estimator=DecisionTreeClassifier(random_state=10),
    # param_grid contains the combinations of hyperparameters to test
    param_grid=dt_param_grid,
    # cv=5 means 5-fold cross validation will be used
    cv=5,
    # scoring defines the evaluation metric used during tuning
    scoring="accuracy",
    # n_jobs=-1 uses all CPU cores to speed up computation. from 30s to less than 10s!
    n_jobs=-1
)

# fit the grid search on the training data
#GridSearchCV will train multiple trees using different parameter combinations
dt_grid.fit(X_train, y_train)

# print the best hyperparameters found by the grid search
print("Best Decision Tree Parameters:", dt_grid.best_params_)

Best Decision Tree Parameters: {'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 5}


In [16]:
# store the best tuned Decision Tree model
best_dt_model = dt_grid.best_estimator_

# predict the loan status values for the test set using the tuned model
y_pred_best_dt = best_dt_model.predict(X_test)

# calculate and print the tuned model accuracy
print("Tuned Decision Tree Accuracy:", accuracy_score(y_test, y_pred_best_dt))

# print the evaluation metrics for the tuned model
print("\nTuned Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_best_dt))

# print the confusion matrix for the tuned Decision Tree model
print("\nTuned Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_dt))

Tuned Decision Tree Accuracy: 0.9225469496610734

Tuned Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95      6999
           1       0.90      0.74      0.81      2000

    accuracy                           0.92      8999
   macro avg       0.91      0.86      0.88      8999
weighted avg       0.92      0.92      0.92      8999


Tuned Decision Tree Confusion Matrix:
[[6829  170]
 [ 527 1473]]


In [17]:
# store the baseline decision tree accuracy
baseline_dt_accuracy = accuracy_score(y_test, y_pred_dt)

# store the tuned decision tree accuracy
tuned_dt_accuracy = accuracy_score(y_test, y_pred_best_dt)

# print the baseline and tuned decision tree accuracies for comparison
print("Baseline Decision Tree Accuracy:", baseline_dt_accuracy)
print("Tuned Decision Tree Accuracy:", tuned_dt_accuracy)

Baseline Decision Tree Accuracy: 0.9051005667296367
Tuned Decision Tree Accuracy: 0.9225469496610734


In [18]:
# create a df containing feature importance values from the tuned decision tree model
dt_feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_dt_model.feature_importances_
})

# sort the features from most important to least important
dt_feature_importance = dt_feature_importance.sort_values(by="Importance", ascending=False)

# display the top 10 most important features
dt_feature_importance.head(10)

,Feature,Importance
21,previous_loan_defaults_on_file_Yes,0.511074
5,loan_percent_income,0.141699
4,loan_int_rate,0.136305
1,person_income,0.080700
15,person_home_ownership_RENT,0.059052
14,person_home_ownership_OWN,0.013961
7,credit_score,0.012146
20,loan_intent_VENTURE,0.010827
17,loan_intent_HOMEIMPROVEMENT,0.010194
18,loan_intent_MEDICAL,0.005777


## Logistic Regression Model

In [19]:
# create the baseline logistic regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# train the baseline logistic regression model using the scaled training data
lr_model.fit(X_train_scaled, y_train)

# predict the loan status values for the scaled test set
y_pred_lr = lr_model.predict(X_test_scaled)

In [20]:
# calculate and print the baseline Logistic Regression accuracy
print("Baseline Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

# print the evaluation metrics 
print("\nBaseline Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

# print the confusion matrix
print("\nBaseline Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

Baseline Logistic Regression Accuracy: 0.9006556284031559

Baseline Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      6999
           1       0.78      0.77      0.78      2000

    accuracy                           0.90      8999
   macro avg       0.86      0.85      0.86      8999
weighted avg       0.90      0.90      0.90      8999


Baseline Logistic Regression Confusion Matrix:
[[6561  438]
 [ 456 1544]]


In [21]:
# define the hyperparameter grid for Logistic Regression
lr_param_grid = {
    # penalty controls regularization type
    #"penalty": ["l2"],
    # C controls regularization strength
    # smaller values increase regularization
    "C": [0.01, 0.1, 1, 10, 100],
    # solver defines the optimization algorithm used
    "solver": ["liblinear", "lbfgs"]
}

# create the GridSearchCV object for Logistic Regression tuning
lr_grid = GridSearchCV(
    # estimator is the base decision tree model
    estimator=LogisticRegression(random_state=10, max_iter=1000),
    # param_grid contains the combinations of hyperparameters to test
    param_grid=lr_param_grid,
     # cv=5 means 5-fold cross validation will be used
    cv=5,
    # scoring defines the evaluation metric used during tuning
    scoring="accuracy",
    # n_jobs=-1 uses all CPU cores to speed up computation
    n_jobs=-1
)

# fit the grid search on the scaled training data
lr_grid.fit(X_train_scaled, y_train)

# print the best logistic regression hyperparameters found by the grid search
print("Best Logistic Regression Parameters:", lr_grid.best_params_)

Best Logistic Regression Parameters: {'C': 0.1, 'solver': 'lbfgs'}


In [22]:
# store the best tuned logistic regression model
best_lr_model = lr_grid.best_estimator_

# predict the loan status values for the scaled test set using the tuned model
y_pred_best_lr = best_lr_model.predict(X_test_scaled)

# calculate and print the tuned logistic regression accuracy
print("Tuned Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_best_lr))

# print the classification report 
print("\nTuned Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_best_lr))

# print the confusion matrix 
print("\nTuned Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_lr))

Tuned Logistic Regression Accuracy: 0.9006556284031559

Tuned Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      6999
           1       0.78      0.77      0.78      2000

    accuracy                           0.90      8999
   macro avg       0.86      0.85      0.86      8999
weighted avg       0.90      0.90      0.90      8999


Tuned Logistic Regression Confusion Matrix:
[[6562  437]
 [ 457 1543]]


In [23]:
# create a df containing logistic regression coefficients
lr_coefficients = pd.DataFrame({
    "Feature": X.columns, # feature names
    "Coefficient": best_lr_model.coef_[0] # coefficients learned by the logistic regression model
})

# compute the odds ratio for each feature using the exponential function
lr_coefficients["Odds_Ratio"] = np.exp(lr_coefficients["Coefficient"])

# compute the absolute value of each coefficient to rank feature influence
lr_coefficients["Absolute_Coefficient"] = lr_coefficients["Coefficient"].abs()

# sort the coefficients by absolute value from largest to smallest
lr_coefficients = lr_coefficients.sort_values(by="Absolute_Coefficient", ascending=False)

# display the top 10 most influentiallLogistic regression features
lr_coefficients.head(10)

,Feature,Coefficient,Odds_Ratio,Absolute_Coefficient
21,previous_loan_defaults_on_file_Yes,-3.468698,0.031158,3.468698
5,loan_percent_income,1.389754,4.013863,1.389754
4,loan_int_rate,0.969027,2.635378,0.969027
3,loan_amnt,-0.682161,0.505523,0.682161
20,loan_intent_VENTURE,-0.449702,0.637818,0.449702
7,credit_score,-0.421532,0.656041,0.421532
15,person_home_ownership_RENT,0.374311,1.453989,0.374311
16,loan_intent_EDUCATION,-0.363476,0.695255,0.363476
14,person_home_ownership_OWN,-0.340296,0.711559,0.340296
19,loan_intent_PERSONAL,-0.259586,0.771371,0.259586


## Model Comparison

In [24]:
# create a comparison table that summarizes the performance of all models
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Tuned Decision Tree",
        "Baseline Logistic Regression",
        "Tuned Logistic Regression"
    ],
    # accuracy values for each model
    "Accuracy": [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_best_dt),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_best_lr)
    ],
    # precision values
    "Precision": [
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_best_dt),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_best_lr)
    ],
    #recall values
    "Recall": [
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_best_dt),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_best_lr)
    ],
    #f1 scores
    "F1 Score": [
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_best_dt),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_best_lr)
    ]
})

# display the model comparison table.
results

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline Decision Tree,0.905101,0.781434,0.7955,0.788404
1,Tuned Decision Tree,0.922547,0.896531,0.7365,0.808674
2,Baseline Logistic Regression,0.900656,0.779011,0.7720,0.775490
3,Tuned Logistic Regression,0.900656,0.779293,0.7715,0.775377
